# EY Water Quality — Target-Specific Feature Engineering (EC + DRP)

**Domain-driven features for each target, based on your running doc's science:**

**EC** = dissolved ions/salinity → driven by geology, evaporation, temperature, seasonal dilution/concentration
- SWIR-based salinity indices (from salinity mapping literature)
- PET × season interactions (evapoconcentration)
- Ion chemistry composites from DWS
- Soil/geology interactions

**DRP** = phosphorus/eutrophication → driven by ag runoff, sewage, high-flow flushing events
- Agriculture × wet season interactions (fertilizer flush)
- Human pressure composites (pop density × urban land)
- Runoff proxies (NDMI/MNDWI × season)
- Algal bloom spectral proxies (nir/green)

Alkalinity is left as-is per your request.

In [ ]:
# !pip install optuna lightgbm catboost shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import os
import optuna
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
import warnings
import matplotlib.pyplot as plt
import json

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
EPS = 1e-6  # safe division constant

In [ ]:
class Config:
    BASE_DIR = './data'  # <<< CHANGE THIS
    SEED = 85
    N_FOLDS = 5
    N_OPTUNA_TRIALS = 30
    TARGETS = [
        'Total Alkalinity',
        'Electrical Conductance',
        'Dissolved Reactive Phosphorus'
    ]

In [ ]:
# ==========================================
# SHARED ENGINEERING (applied to all targets)
# ==========================================
def engineer_shared(df):
    """Features that help all 3 targets. Run first."""
    d = df.copy()
    
    # --- Remote sensing band ratios (your running doc Rank 4) ---
    if all(c in d.columns for c in ['nir', 'green', 'swir16', 'swir22']):
        d['nir_green_ratio'] = d['nir'] / (d['green'] + EPS)
        d['swir16_nir_ratio'] = d['swir16'] / (d['nir'] + EPS)
        d['swir22_nir_ratio'] = d['swir22'] / (d['nir'] + EPS)
        d['swir16_green_ratio'] = d['swir16'] / (d['green'] + EPS)
        d['nir_minus_green'] = d['nir'] - d['green']
        d['swir_diff'] = d['swir16'] - d['swir22']
    
    # --- Log-transform skewed features ---
    for feat in ['Popdens_00', 'SOC', 'dist_km']:
        if feat in d.columns:
            d[f'log_{feat}'] = np.log1p(d[feat])
    
    return d

In [ ]:
# ==========================================
# EC-SPECIFIC ENGINEERING
# ==========================================
# Domain: EC = dissolved ions. Driven by:
#   - Evaporation/temperature (PET is a strong proxy)
#   - Geology (soil type, lithology)
#   - Seasonal concentration vs dilution
#   - SWIR bands track surface moisture/salt
#   - DWS ion chemistry (Cl, SO4, Na, Mg, Ca)

def engineer_ec(df):
    """EC-specific features. Call AFTER engineer_shared."""
    d = df.copy()
    
    # --- 1. Evapoconcentration proxy (PET × season) ---
    # High PET + dry season = concentrated ions = higher EC
    if all(c in d.columns for c in ['pet', 'wet_season']):
        d['ec_pet_x_dry'] = d['pet'] * (1 - d['wet_season'])  # PET in dry season
        d['ec_pet_x_wet'] = d['pet'] * d['wet_season']        # PET in wet season (dilution)
    
    # PET × month cyclical (captures EC's strong temperature dependence)
    if all(c in d.columns for c in ['pet', 'month_sin', 'month_cos']):
        d['ec_pet_x_msin'] = d['pet'] * d['month_sin']
        d['ec_pet_x_mcos'] = d['pet'] * d['month_cos']
    
    # --- 2. Salinity indices from SWIR (from salinity mapping paper) ---
    # These are the top indices for soil/water salinity detection
    if all(c in d.columns for c in ['swir16', 'swir22', 'nir', 'green']):
        # SI (Salinity Index) = sqrt(swir16 * swir22)
        d['ec_salinity_idx'] = np.sqrt(np.abs(d['swir16'] * d['swir22']))
        # NDSI-like (Normalized Diff Salinity Index)
        d['ec_ndsi'] = (d['swir16'] - d['swir22']) / (d['swir16'] + d['swir22'] + EPS)
        # Brightness index (tracks bare/salty surfaces)
        d['ec_brightness'] = np.sqrt(d['green']**2 + d['nir']**2 + d['swir16']**2)
    
    # --- 3. DWS ion chemistry composite ---
    # EC is literally the sum of dissolved ions. If we have nearby DWS measurements,
    # their ion chemistry is extremely predictive
    ion_cols = ['dws_Ca', 'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4']
    present_ions = [c for c in ion_cols if c in d.columns]
    if len(present_ions) >= 2:
        # Total ion load (crude but effective)
        d['ec_total_ions'] = d[present_ions].sum(axis=1)
        # Dominant anion ratio (Cl vs SO4 — indicates source: marine vs mining)
        if 'dws_Cl' in d.columns and 'dws_SO4' in d.columns:
            d['ec_cl_so4_ratio'] = d['dws_Cl'] / (d['dws_SO4'] + EPS)
        # Cation ratio (Ca+Mg vs Na — indicates weathering type)
        if all(c in d.columns for c in ['dws_Ca', 'dws_Mg', 'dws_Na']):
            d['ec_camg_na_ratio'] = (d['dws_Ca'] + d['dws_Mg']) / (d['dws_Na'] + EPS)
    
    # --- 4. Geology/soil interactions ---
    # Clay soils → higher EC (ionizable minerals). Granite → lower EC.
    # Lithology columns: sc, ss, su, mt, va, vb, vi, pa, pb, pi
    # sc = sedimentary carbonate (high EC), ss = sedimentary siliciclastic,
    # mt = metamorphic, va/vb/vi = volcanic
    if 'Soil_pH' in d.columns and 'pet' in d.columns:
        d['ec_soilpH_x_pet'] = d['Soil_pH'] * d['pet']  # alkaline soil + evaporation = ions
    
    if 'Soil_wetness' in d.columns:
        # Dry soil + high PET = salt accumulation at surface
        if 'pet' in d.columns:
            d['ec_drysoil_pet'] = (1 - d['Soil_wetness']) * d['pet']
    
    # Sedimentary carbonate fraction → strong EC predictor
    if 'sc' in d.columns:
        if 'pet' in d.columns:
            d['ec_sc_x_pet'] = d['sc'] * d['pet']  # carbonate geology + evaporation
    
    # --- 5. Human pressure × season ---
    # Agriculture/mining return flows are seasonal
    if 'GLC_Managed' in d.columns and 'wet_season' in d.columns:
        d['ec_agri_x_wet'] = d['GLC_Managed'] * d['wet_season']  # ag washoff
    if 'GLC_Artificial' in d.columns and 'wet_season' in d.columns:
        d['ec_urban_x_wet'] = d['GLC_Artificial'] * d['wet_season']  # urban runoff
    
    # --- 6. DWS EC reliability weighting ---
    if all(c in d.columns for c in ['dws_EC', 'dws_dist_km', 'dws_days_diff']):
        # Distance-decayed DWS EC (trust DWS less when far away)
        d['ec_dws_decay'] = d['dws_EC'] / (1 + d['dws_dist_km'] * 0.1 + np.abs(d['dws_days_diff']) * 0.01)
    
    return d

In [ ]:
# ==========================================
# DRP-SPECIFIC ENGINEERING
# ==========================================
# Domain: DRP = phosphorus. Driven by:
#   - Agricultural runoff (fertilizer flushing during wet season)
#   - Sewage/wastewater (population, urban land)
#   - High-flow events (rainfall → runoff → P mobilization)
#   - Bimodal distribution (background vs event-driven spikes)
#   - Algal bloom indicators (nir/green as chlorophyll proxy)

def engineer_drp(df):
    """DRP-specific features. Call AFTER engineer_shared."""
    d = df.copy()
    
    # --- 1. Agricultural runoff proxies ---
    # DRP in summer is ~4x higher than winter (domain lit)
    # Fertilizer gets flushed during wet season from managed land
    if 'GLC_Managed' in d.columns:
        if 'wet_season' in d.columns:
            d['drp_agri_x_wet'] = d['GLC_Managed'] * d['wet_season']  # THE key interaction
            d['drp_agri_x_dry'] = d['GLC_Managed'] * (1 - d['wet_season'])
        if 'pet' in d.columns:
            d['drp_agri_x_pet'] = d['GLC_Managed'] * d['pet']  # more evapotranspiration = growing season = fertilizer use
        if 'NDMI' in d.columns:
            d['drp_agri_x_ndmi'] = d['GLC_Managed'] * d['NDMI']  # wet ag land = active runoff
    
    # --- 2. Sewage / human pressure composites ---
    if 'GLC_Artificial' in d.columns and 'Popdens_00' in d.columns:
        # Combined human pressure index
        d['drp_human_pressure'] = np.log1p(d['Popdens_00']) * d['GLC_Artificial']
    if 'GLC_Artificial' in d.columns and 'wet_season' in d.columns:
        d['drp_urban_x_wet'] = d['GLC_Artificial'] * d['wet_season']  # sewage overflow during rain
    if 'Popdens_00' in d.columns and 'wet_season' in d.columns:
        d['drp_pop_x_wet'] = np.log1p(d['Popdens_00']) * d['wet_season']
    
    # --- 3. Runoff / flushing event proxies ---
    # NDMI and MNDWI spike during high-flow events that mobilize P
    if 'NDMI' in d.columns and 'MNDWI' in d.columns:
        # Water presence + moisture = active flow
        d['drp_water_signal'] = d['MNDWI'] * d['NDMI']  # both high = active water body
    if 'MNDWI' in d.columns and 'wet_season' in d.columns:
        d['drp_mndwi_x_wet'] = d['MNDWI'] * d['wet_season']
    if 'NDMI' in d.columns and 'wet_season' in d.columns:
        d['drp_ndmi_x_wet'] = d['NDMI'] * d['wet_season']
    
    # PET as inverse runoff proxy (low PET = cool/wet = more runoff)
    if 'pet' in d.columns:
        d['drp_inv_pet'] = 1.0 / (d['pet'] + EPS)  # low PET → more runoff potential
        if 'GLC_Managed' in d.columns:
            d['drp_agri_lowpet'] = d['GLC_Managed'] * d['drp_inv_pet']  # ag land + high runoff
    
    # --- 4. Algal bloom / eutrophication proxies ---
    # High DRP → algal blooms → changes in spectral signature
    # nir/green ratio is a crude chlorophyll-a proxy
    if all(c in d.columns for c in ['nir', 'green']):
        d['drp_chl_proxy'] = d['nir'] / (d['green'] + EPS)  # chlorophyll-a proxy
        # Green peak relative to NIR (turbid/eutrophic water)
        d['drp_green_dominance'] = d['green'] / (d['nir'] + d['green'] + EPS)
    if all(c in d.columns for c in ['nir', 'swir16']):
        # Turbidity-like index (high nir, low swir = turbid water)
        d['drp_turbidity_proxy'] = (d['nir'] - d['swir16']) / (d['nir'] + d['swir16'] + EPS)
    
    # --- 5. Aquatic vegetation interaction ---
    # Areas with more aquatic vegetation may trap or release P
    if 'GLC_Aquatic_Veg' in d.columns:
        if 'wet_season' in d.columns:
            d['drp_aquaveg_x_wet'] = d['GLC_Aquatic_Veg'] * d['wet_season']
        if 'pet' in d.columns:
            d['drp_aquaveg_x_pet'] = d['GLC_Aquatic_Veg'] * d['pet']
    
    # --- 6. DWS phosphorus reliability-weighted ---
    if all(c in d.columns for c in ['dws_P_modified', 'dws_dist_km', 'dws_days_diff']):
        d['drp_dws_decay'] = d['dws_P_modified'] / (1 + d['dws_dist_km'] * 0.1 + np.abs(d['dws_days_diff']) * 0.01)
    
    # --- 7. Seasonal magnitude (DRP is 4x higher summer vs winter) ---
    if 'month' in d.columns:
        # Peak flushing months in SA: Dec-Mar (summer rainy season)
        d['drp_peak_flush'] = d['month'].isin([12, 1, 2, 3]).astype(int)
        # Transition months (Oct-Nov) when first rains hit dry fertilized land
        d['drp_first_flush'] = d['month'].isin([10, 11]).astype(int)
    
    return d

In [ ]:
# ==========================================
# LOAD DATA
# ==========================================
print("Loading...")
train_full = pd.read_csv(f'{Config.BASE_DIR}/train_ALL+reliability.csv')
test_full = pd.read_csv(f'{Config.BASE_DIR}/test_ALL+reliability.csv')

# Spatial clusters
stations = train_full.groupby(['Latitude', 'Longitude']).size().reset_index()
kmeans = KMeans(n_clusters=Config.N_FOLDS, random_state=Config.SEED, n_init=10)
stations['spatial_cluster'] = kmeans.fit_predict(stations[['Latitude', 'Longitude']])
train_full = train_full.merge(
    stations[['Latitude', 'Longitude', 'spatial_cluster']],
    on=['Latitude', 'Longitude'], how='left'
)

# Apply shared engineering
train_shared = engineer_shared(train_full)
test_shared = engineer_shared(test_full)

print(f"Train: {train_shared.shape}, Test: {test_shared.shape}")

In [ ]:
# ==========================================
# BUILD PER-TARGET DATASETS
# ==========================================
ignore_cols = Config.TARGETS + [
    'Latitude', 'Longitude', 'STAT_ID', 'Sample Date',
    'spatial_cluster', 'geometry', '_merge_terra',
    '_merge_landsat', 'Latitude_glorich',
    'Longitude_glorich', 'date', 'dws_1st', 'Impute_Method',
    'P_modified_same', 'dws_P_reliability'
]

def prune_features(X, corr_thresh=0.95):
    X = X.loc[:, X.nunique() > 1]
    # Handle any remaining non-numeric
    X = X.select_dtypes(include=[np.number])
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > corr_thresh)]
    print(f"  Corr pruning: dropped {len(to_drop)}/{len(X.columns)} features")
    return X.drop(columns=to_drop)

# Alkalinity — shared features only (no target-specific engineering)
train_alk = train_shared.copy()
test_alk = test_shared.copy()

# EC — add EC-specific features
train_ec = engineer_ec(train_shared)
test_ec = engineer_ec(test_shared)

# DRP — add DRP-specific features
train_drp = engineer_drp(train_shared)
test_drp = engineer_drp(test_shared)

# Build feature matrices per target
datasets = {}
for target, tr, te in [
    ('Total Alkalinity', train_alk, test_alk),
    ('Electrical Conductance', train_ec, test_ec),
    ('Dissolved Reactive Phosphorus', train_drp, test_drp),
]:
    print(f"\n--- {target} ---")
    feats = [c for c in tr.columns if c not in ignore_cols]
    X = prune_features(tr[feats])
    X_te = te[[c for c in X.columns if c in te.columns]]
    
    # Check for missing cols in test
    missing = set(X.columns) - set(X_te.columns)
    if missing:
        print(f"  WARNING: {len(missing)} features missing in test: {missing}")
        X = X.drop(columns=list(missing))
        X_te = te[X.columns]
    
    datasets[target] = {'X_train': X, 'X_test': X_te}
    print(f"  Final features: {X.shape[1]}")
    
    # Show target-specific features
    if target == 'Electrical Conductance':
        ec_feats = [c for c in X.columns if c.startswith('ec_')]
        print(f"  EC-specific features ({len(ec_feats)}): {ec_feats}")
    elif target == 'Dissolved Reactive Phosphorus':
        drp_feats = [c for c in X.columns if c.startswith('drp_')]
        print(f"  DRP-specific features ({len(drp_feats)}): {drp_feats}")

In [ ]:
# ==========================================
# OPTUNA OBJECTIVE
# ==========================================
def lgbm_objective(trial, X, y, groups, target_name):
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': Config.SEED,
        'n_jobs': -1,
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample_freq': 1,
    }
    
    # DRP-specific: try Huber loss (robust to outliers/spikes)
    if 'Phosphorus' in target_name:
        loss_type = trial.suggest_categorical('loss_type', ['rmse', 'huber'])
        if loss_type == 'huber':
            param['objective'] = 'huber'
            param['alpha'] = trial.suggest_float('huber_alpha', 0.5, 20.0)
    
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []
    
    for tr_idx, va_idx in gkf.split(X, y, groups=groups):
        model = lgb.LGBMRegressor(**param)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
        )
        preds = model.predict(X.iloc[va_idx])
        if 'Phosphorus' in target_name:
            scores.append(r2_score(np.expm1(y.iloc[va_idx]), np.expm1(preds)))
        else:
            scores.append(r2_score(y.iloc[va_idx], preds))
    
    return np.mean(scores)

In [ ]:
# ==========================================
# TRAIN ALL 3 TARGETS (each with own features)
# ==========================================
final_preds = {}
models = {}
best_params_all = {}
cv_scores = {}
groups = train_shared['spatial_cluster']

for target in Config.TARGETS:
    print(f"\n{'='*55}")
    print(f"  {target}")
    print(f"  Features: {datasets[target]['X_train'].shape[1]}")
    print(f"{'='*55}")
    
    X = datasets[target]['X_train']
    X_te = datasets[target]['X_test']
    
    y = train_shared[target].copy()
    y_train = np.log1p(y) if target == 'Dissolved Reactive Phosphorus' else y
    y_train.name = target
    
    # Optuna
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial: lgbm_objective(trial, X, y_train, groups, target),
        n_trials=Config.N_OPTUNA_TRIALS,
        show_progress_bar=True
    )
    
    best_params_all[target] = study.best_params
    cv_scores[target] = study.best_value
    print(f"  Best CV R²: {study.best_value:.4f}")
    
    # Final model
    final_params = {k: v for k, v in study.best_params.items() if k not in ['loss_type', 'huber_alpha']}
    # Handle Huber if selected for DRP
    if study.best_params.get('loss_type') == 'huber':
        final_params['objective'] = 'huber'
        final_params['alpha'] = study.best_params.get('huber_alpha', 9.0)
    else:
        final_params['objective'] = 'regression'
    final_params.update({
        'n_estimators': 2000,
        'metric': 'rmse',
        'verbosity': -1,
        'seed': Config.SEED,
        'n_jobs': -1,
    })
    
    final_model = lgb.LGBMRegressor(**final_params)
    final_model.fit(X, y_train)
    models[target] = final_model
    
    # Save
    final_model.booster_.save_model(f'{Config.BASE_DIR}/lgbm_targeteng_{target.replace(" ", "_")}.txt')
    
    # Predict
    preds = final_model.predict(X_te)
    if target == 'Dissolved Reactive Phosphorus':
        preds = np.expm1(preds)
    final_preds[target] = np.clip(preds, 0, None)

print("\n" + "="*55)
print("  CV SCORES")
print("="*55)
for t, s in cv_scores.items():
    print(f"  {t}: {s:.4f}")
print(f"  MEAN R²: {np.mean(list(cv_scores.values())):.4f}")

In [ ]:
# ==========================================
# FEATURE IMPORTANCE — which target-specific features helped?
# ==========================================
for target in ['Electrical Conductance', 'Dissolved Reactive Phosphorus']:
    model = models[target]
    X = datasets[target]['X_train']
    prefix = 'ec_' if 'Conductance' in target else 'drp_'
    
    fi = pd.DataFrame({
        'Feature': X.columns,
        'Gain': model.booster_.feature_importance(importance_type='gain'),
    }).sort_values('Gain', ascending=False)
    fi['Gain_pct'] = fi['Gain'] / fi['Gain'].max() * 100
    
    # Show all features, highlighting target-specific ones
    fi['is_target_specific'] = fi['Feature'].str.startswith(prefix)
    
    print(f"\n{'='*55}")
    print(f"  {target} — Top 20 Features")
    print(f"{'='*55}")
    top20 = fi.head(20)
    for _, row in top20.iterrows():
        marker = ' ★' if row['is_target_specific'] else ''
        print(f"  {row['Gain_pct']:6.1f}%  {row['Feature']}{marker}")
    
    # Summary: how do target-specific features rank?
    target_feats = fi[fi['is_target_specific']]
    print(f"\n  Target-specific features ({len(target_feats)}):")
    print(f"  Mean rank: {fi.index[fi['is_target_specific']].to_list()}")
    print(f"  Total gain share: {target_feats['Gain'].sum() / fi['Gain'].sum() * 100:.1f}%")
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#FF6B35' if ts else '#2196F3' for ts in top20['is_target_specific'][::-1]]
    ax.barh(top20['Feature'][::-1], top20['Gain_pct'][::-1], color=colors)
    ax.set_xlabel('Gain (% of top)')
    ax.set_title(f'{target}\nOrange = target-specific features')
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# SUBMISSION
# ==========================================
submission = pd.DataFrame({
    'Latitude': test_full['Latitude'],
    'Longitude': test_full['Longitude'],
    'Sample Date': test_full['Sample Date'],
    'Total Alkalinity': final_preds['Total Alkalinity'],
    'Electrical Conductance': final_preds['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': final_preds['Dissolved Reactive Phosphorus']
})

# Hybrid DWS override for TAL
dws_tal = test_full['dws_TAL'].values
submission['Total Alkalinity'] = np.where(
    np.isfinite(dws_tal), dws_tal, submission['Total Alkalinity']
)

submission.to_csv(f'{Config.BASE_DIR}/submission_lgbm_targeteng.csv', index=False)
print("Submission saved.")

In [ ]:
# ==========================================
# ENSEMBLE with CatBoost if available
# ==========================================
catboost_path = f'{Config.BASE_DIR}/submission_reliabilityOfDWS_temp1_all_features.csv'
if os.path.exists(catboost_path):
    sub_cat = pd.read_csv(catboost_path)
    
    w_cat, w_lgb = 0.5, 0.5
    ensemble = sub_cat[['Latitude', 'Longitude', 'Sample Date']].copy()
    for target in Config.TARGETS:
        ensemble[target] = np.clip(
            w_cat * sub_cat[target] + w_lgb * submission[target], 0, None
        )
    ensemble.to_csv(f'{Config.BASE_DIR}/submission_ensemble_targeteng.csv', index=False)
    print("Ensemble saved!")
    
    print("\nDivergence:")
    for target in Config.TARGETS:
        diff = np.abs(sub_cat[target] - submission[target]).mean()
        print(f"  {target}: {diff:.2f}")
else:
    print(f"CatBoost sub not found at {catboost_path}")

In [ ]:
# ==========================================
# SAVE RESULTS
# ==========================================
results = {
    'model': 'LightGBM + target-specific engineering',
    'cv_scores': {k: float(v) for k, v in cv_scores.items()},
    'mean_r2': float(np.mean(list(cv_scores.values()))),
    'n_features': {t: datasets[t]['X_train'].shape[1] for t in Config.TARGETS},
    'ec_specific_features': [c for c in datasets['Electrical Conductance']['X_train'].columns if c.startswith('ec_')],
    'drp_specific_features': [c for c in datasets['Dissolved Reactive Phosphorus']['X_train'].columns if c.startswith('drp_')],
}
with open(f'{Config.BASE_DIR}/lgbm_targeteng_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(json.dumps(results, indent=2, default=str))